## I want to investigate whether FBA can recapitulate WC behavior when adjusting objetive weights.

In [2]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [9]:
# import sim
time = '600'
date = '2026-02-10'
experiment = 'homeo_kinetics_3.15E-3_2'
condition = 'basal'
entry = f'{experiment}_{time}_{date}'
folder = f'out/objective_weight/{condition}/{entry}/'

output = np.load(folder + '0_output.npy',allow_pickle='TRUE').item()
output_combo = output['agents']['0']
fba_combo = output_combo['listeners']['fba_results']
bulk = pd.DataFrame(output_combo['bulk'])
f = open(folder + 'agent_steps.pkl', 'rb')
agent = dill.load(f)
f.close()

metabolism = agent['ecoli-metabolism-redux-classic']

In [10]:
# import sim
time = '600'
date = '2026-02-10'
experiment = 'homeostatic_only_2'
condition = 'basal'
entry = f'{experiment}_{time}_{date}'
folder = f'out/objective_weight/{condition}/{entry}/'

output = np.load(folder + '0_output.npy',allow_pickle='TRUE').item()
output_h = output['agents']['0']
fba_h_only = output_h['listeners']['fba_results']
f = open(folder + 'agent_steps.pkl', 'rb')
agent = dill.load(f)
f.close()

metabolism_h_only = agent['ecoli-metabolism-redux-classic']

In [11]:
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
metabolites = metabolism.metabolite_names.copy()
counts_to_molar_h = output_h['listeners']['enzyme_kinetics']['counts_to_molar']
counts_to_molar_combo = output_combo['listeners']['enzyme_kinetics']['counts_to_molar']

homeostatic_dm_targets =  pd.DataFrame(fba_combo['target_homeostatic_dmdt'],
                                       columns=metabolism.homeostatic_metabolites).mul(counts_to_molar_combo, axis=0).mean(axis=0) # in conc
homeostatic_metabolite_counts = pd.DataFrame(fba_combo['homeostatic_metabolite_counts'],
                                             columns=metabolism.homeostatic_metabolites).mul(counts_to_molar_combo, axis=0).mean(axis=0) # in conc
maintenance = pd.DataFrame(fba_combo["maintenance_target"][1:],
                           columns=['maintenance_reaction']).mul(counts_to_molar_combo[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba_combo["target_kinetic_fluxes"],
                       columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar_combo, axis=0).mean(axis=0)

binary_kinetic_idx = fba_combo['binary_kinetic_idx'][1:]

print(len(homeostatic_dm_targets))
len(homeostatic_metabolite_counts)

172


172

In [12]:
FREE_RXNS = [
    "TRANS-RXN-145",
    "TRANS-RXN0-545",
    "TRANS-RXN0-474",
    "ATPSYN-RXN (reverse)",
]

def test_NetworkFlowModel(weights, solver_choice = cp.GLOP,
                          homeostatic_metabolite_counts = homeostatic_metabolite_counts,
                          homeostatic_dm_targets = homeostatic_dm_targets,
                          kinetic=kinetic,
                          binary_kinetics_idx=None,
                          maintenance=maintenance,
                          ):
    model = NetworkFlowModel(
            stoich_arr=stoichiometry,
            metabolites=metabolites,
            reactions=reaction_names,
            homeostatic_metabolites=metabolism.homeostatic_metabolites,
            kinetic_reactions=metabolism.kinetic_constraint_reactions,
            free_reactions=FREE_RXNS)
    model.set_up_exchanges(exchanges=metabolism.exchange_molecules, uptakes=metabolism.allowed_exchange_uptake)
    solution: FlowResult = model.solve(
            homeostatic_concs=homeostatic_metabolite_counts, # in conc
            homeostatic_dm_targets=np.array(list(dict(homeostatic_dm_targets).values())), # conc
            maintenance_target=maintenance, # conc range
            kinetic_targets=np.array(list(dict(kinetic).values())), # conc range
            binary_kinetic_idx=binary_kinetics_idx,
            # force_flow_idx=force_reaction_idx,
            objective_weights=weights, #same
            upper_flux_bound= 100, # increase to 10^9 because notebook runs FlowResult using Counts, WC runs using conc.
            solver=solver_choice) #SCS. ECOS, MOSEK
    return solution.objective, solution.homeostatic_term, solution.kinetics_term, solution.velocities, solution.dm_dt

In [13]:
def met_homeo(MOI, dmdt):
    ddt = dmdt.loc[MOI]
    return np.abs(ddt-homeostatic_dm_targets.loc[MOI])/homeostatic_metabolite_counts[MOI]

### I want to test whether choosing a weight can recapitulate a whole cell sim behavior

In [14]:
binary_kinetic_idx

[[[70,
   135,
   136,
   272,
   273,
   434,
   435,
   436,
   550,
   553,
   728,
   1040,
   1181,
   1245,
   1246,
   1311,
   1329,
   1330,
   1386,
   1387,
   1499,
   1500,
   1501,
   1693,
   1694,
   1890,
   2088,
   2296,
   2317,
   2599,
   2602,
   2603,
   2850,
   3169,
   3215,
   3355,
   3395,
   3396,
   3397,
   3398,
   3399,
   3400,
   3464,
   3486,
   3607,
   3614,
   3733,
   6946,
   6949,
   6964,
   6965,
   6974,
   7245,
   7321,
   7322,
   7323,
   7324,
   7325,
   7326,
   7327,
   7444,
   7445,
   7446,
   7447,
   7448,
   7449,
   7450,
   7451,
   7452,
   7453,
   7494,
   7495,
   7496,
   7497,
   7498,
   7499,
   7500,
   7501,
   7502,
   7503,
   7504,
   7505,
   7675,
   7676,
   7677,
   8055,
   8056,
   8057,
   8058,
   8059,
   8060,
   8061,
   8062,
   8063,
   8064,
   8065,
   8066,
   8067,
   8102,
   9394,
   9395]],
 [[70,
   135,
   136,
   272,
   273,
   434,
   435,
   436,
   550,
   553,
   728,
   1040,
   11

In [15]:
fba = fba_combo.copy()
counts_to_molar = counts_to_molar_combo.copy()

homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0)

binary_kinetic_idx = fba['binary_kinetic_idx'][1:]

In [16]:
fba = fba_h_only.copy()
counts_to_molar = counts_to_molar_h.copy()

homeostatic_dm_targets_h =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0)
homeostatic_metabolite_counts_h = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0)
maintenance_h = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0)
kinetic_h = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0)

binary_kinetic_idx_h = fba['binary_kinetic_idx'][1:]

In [14]:
reaction = "UDP-NACMURALA-GLU-LIG-RXN"
reaction_idx = reaction_names.index("UDP-NACMURALA-GLU-LIG-RXN")


count = 0
for idx_time in binary_kinetic_idx_h:
    if reaction_idx in idx_time[0]:
        count += 1
        # print('reaction not available')
count

184

In [67]:
objective_weights = {
    "kinetics": 0.00315,
    "homeostatic": 1
}

obj_val_arr = []
home_term_arr = []
kin_term_arr = []
v_arr = np.zeros((600, len(reaction_names)))
dmdt_arr = np.zeros((600, len(metabolites)))

for timestep in range(1, len(homeostatic_dm_targets)):
    dm_target = homeostatic_dm_targets_h.loc[timestep]
    m_count = homeostatic_metabolite_counts.loc[timestep]
    maintenance_ts = maintenance_h.loc[timestep-1]
    kinetics_ts = kinetic_h.loc[timestep]
    kinetics_idx = binary_kinetic_idx_h[timestep-1]

    obj_val, home_term, kin_term, v, dmdt = test_NetworkFlowModel(
        objective_weights,
        homeostatic_metabolite_counts = m_count,
        homeostatic_dm_targets = dm_target,
        kinetic=kinetics_ts,
        maintenance=maintenance_ts,
        binary_kinetics_idx=kinetics_idx)

    obj_val_arr.append(obj_val)
    home_term_arr.append(home_term)
    kin_term_arr.append(kin_term)
    v_arr[timestep-1,:] = v
    dmdt_arr[timestep-1,:] = dmdt

In [68]:
df_obj = pd.DataFrame({
    "timestep": range(len(obj_val_arr)),
    "objective": obj_val_arr,
    "home_term": home_term_arr * objective_weights["homeostatic"],
    "kin_term": np.asarray(kin_term_arr) * objective_weights["kinetics"],
})

long = df_obj.melt(id_vars="timestep", var_name="term", value_name="value")

fig = px.line(
    long, x="timestep", y="value", color="term",
    title=f"Objective decomposition over time, kinetic weight = {objective_weights["kinetics"]}, param used is all home_only, except home met count"
)
fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()


In [66]:
df_obj = pd.DataFrame({
    "timestep": range(len(obj_val_arr)),
    "objective": obj_val_arr,
    "home_term": home_term_arr * objective_weights["homeostatic"],
    "kin_term": np.asarray(kin_term_arr) * objective_weights["kinetics"],
})

long = df_obj.melt(id_vars="timestep", var_name="term", value_name="value")

fig = px.line(
    long, x="timestep", y="value", color="term",
    title=f"Objective decomposition over time, kinetic weight = {objective_weights["kinetics"]}, param used is all home_only, except home dmdt target"
)
fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()


In [61]:
df_obj = pd.DataFrame({
    "timestep": range(len(obj_val_arr)),
    "objective": obj_val_arr,
    "home_term": home_term_arr * objective_weights["homeostatic"],
    "kin_term": np.asarray(kin_term_arr) * objective_weights["kinetics"],
})

long = df_obj.melt(id_vars="timestep", var_name="term", value_name="value")

fig = px.line(
    long, x="timestep", y="value", color="term",
    title=f"Objective decomposition over time, kinetic weight = {objective_weights["kinetics"]}, param used is home only homeostatic metabolite counts, else is home+kinetic315"
)
fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()


In [58]:
df_obj = pd.DataFrame({
    "timestep": range(len(obj_val_arr)),
    "objective": obj_val_arr,
    "home_term": home_term_arr * objective_weights["homeostatic"],
    "kin_term": np.asarray(kin_term_arr) * objective_weights["kinetics"],
})

long = df_obj.melt(id_vars="timestep", var_name="term", value_name="value")

fig = px.line(
    long, x="timestep", y="value", color="term",
    title=f"Objective decomposition over time, kinetic weight = {objective_weights["kinetics"]}, param used is home only homeostatic metabolite counts, else is home+kinetic315"
)
fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()


In [27]:
# FBA run 600s using homeostatic + kinetic 0.00315 sim on lambda_k = 0.00315 and binary_idx of homeo only

df_obj = pd.DataFrame({
    "timestep": range(len(obj_val_arr)),
    "objective": obj_val_arr,
    "home_term": home_term_arr * objective_weights["homeostatic"],
    "kin_term": np.asarray(kin_term_arr) * objective_weights["kinetics"],
})

long = df_obj.melt(id_vars="timestep", var_name="term", value_name="value")

fig = px.line(
    long, x="timestep", y="value", color="term",
    title="Objective decomposition over time",
)
fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()


In [56]:
# FBA run 600s using homeostatic only sim on lambda_k = 0.00315
df_obj = pd.DataFrame({
    "timestep": range(len(obj_val_arr)),
    "objective": obj_val_arr,
    "home_term": home_term_arr * objective_weights["homeostatic"],
    "kin_term": np.asarray(kin_term_arr) * objective_weights["kinetics"],
})

long = df_obj.melt(id_vars="timestep", var_name="term", value_name="value")

fig = px.line(
    long, x="timestep", y="value", color="term",
    title="Objective decomposition over time",
)
fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()


In [57]:
dmdt_df = pd.DataFrame(dmdt_arr, columns=metabolites)

In [58]:
met = "CPD-12261[p]"

# 1) get index of this metabolite in your metabolite list
met_idx = metabolites.index(met)   # make sure metabolite name matches exactly

# 2) estimated dmdt from your solve (vector over time)
est_dmdt = pd.Series(dmdt_arr[:, met_idx], name="estimated_dmdt")

# 3) target dmdt (vector over time)
tar_dmdt = pd.Series(homeostatic_dm_targets[met], name="target_dmdt")

# 4) unmet need (pick sign you want)
dmdt_diff_interest = (tar_dmdt - est_dmdt).rename("unmet_need")
# or absolute:
# dmdt_diff_interest = (tar_dmdt - est_dmdt).abs().rename("unmet_need")

rxn_name = "UDP-NACMURALA-GLU-LIG-RXN"
rxn_idx = reaction_names.index(rxn_name)

rxn_flux = pd.Series(v_arr[:, rxn_idx],name="estimated_flux")


In [59]:
reaction_catalyst_counts_all_time = pd.DataFrame(fba['reaction_catalyst_counts'][1:], columns=reaction_names)
reaction = "UDP-NACMURALA-GLU-LIG-RXN"
enzyme_timeseries = reaction_catalyst_counts_all_time.loc[:, reaction]
enzyme_name = metabolism.parameters['reaction_catalysts'][reaction]



met = "CPD-12261[p]"  # (you wrote CPD-12251[p] once—use the exact column name you have)
enzyme_label = f"{enzyme_name} (counts)"
need_label = f"{met} unmet need (normalized dmdt diff)"

# --- align indices (time) robustly ---
enz = enzyme_timeseries.copy()
need = dmdt_diff_interest[met].copy() if isinstance(dmdt_diff_interest, pd.DataFrame) else dmdt_diff_interest
pastel = px.colors.qualitative.Pastel
# Ensure both are Series with numeric index
enz = pd.Series(enz.values, index=enz.index, name="enzyme_counts")
need = pd.Series(need.values, index=need.index, name="unmet_need")
reaction_flux = pd.Series(v_arr[:, rxn_idx], name="estimated_flux")

# Inner-join on shared timepoints
df_plot = pd.concat([enz, need, reaction_flux], axis=1, join="inner").dropna()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["enzyme_counts"],
        mode="lines",
        name=enzyme_label,
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["unmet_need"],
        mode="lines",
        name=need_label,
    ),
    secondary_y=True
)

fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["estimated_flux"],
        mode="lines",
        name=f'{reaction} flux',
    ),
    secondary_y=False
)


fig.update_layout(
    title=f"Enzyme availability of UDP-NACMURALA-GLU-LIG-RXN vs unmet need for {met}. λkin = {objective_weights['kinetics']}",
    template="plotly_white",
    legend_title_text="Trace",
    colorway=px.colors.qualitative.Pastel,
    # paper_bgcolor='rgba(255, 0, 0, 0)',
    # plot_bgcolor='rgba(255, 0, 0, 0)',
)

fig.update_xaxes(title_text="Time (s)")
fig.update_yaxes(title_text="Enzyme counts", secondary_y=False)
fig.update_yaxes(title_text="L1_|Target - Estimate|", secondary_y=True)

fig.show()

# check if binary_kinetic_idx differs from sim to sim

In [17]:
# import sim
time = '600'
date = '2026-02-10'
experiment = 'homeostatic_only_2'
condition = 'basal'
entry = f'{experiment}_{time}_{date}'
folder = f'out/objective_weight/{condition}/{entry}/'

output = np.load(folder + '0_output.npy',allow_pickle='TRUE').item()
output = output['agents']['0']
fba = output['listeners']['fba_results']
bulk = pd.DataFrame(output['bulk'])
f = open(folder + 'agent_steps.pkl', 'rb')
agent = dill.load(f)
f.close()

metabolism = agent['ecoli-metabolism-redux-classic']

In [18]:
binary_home_only = fba['binary_kinetic_idx']

In [19]:
time_yes = []
time_no = []
for i in range(600):
    if binary_home_only[i] == binary_kinetic_idx[i]:
        time_yes.append(i)
    else:
        time_no.append(i)

### Yes, different sim has different binary_kinetic_idx

## Check how different the homeostatic metabolite counts are in the two sims. From runnning FBA and sequentially changing the inputs, I figured out that home_met_counts is the most important factor in determining whether the psusdo-sim by FBA will resemble a whole cell sim.

In [20]:
np.all(homeostatic_metabolite_counts_h == homeostatic_metabolite_counts, axis=0)

2-3-DIHYDROXYBENZOATE[c]    False
2-KETOGLUTARATE[c]          False
2-PG[c]                     False
2K-4CH3-PENTANOATE[c]       False
4-AMINO-BUTYRATE[c]         False
                            ...  
NA+[p]                      False
OXYGEN-MOLECULE[p]          False
FE+3[p]                     False
CA+2[p]                     False
Pi[p]                       False
Length: 172, dtype: bool

In [21]:
# Calculate RMS difference for each metabolite (column)
rms_differences = np.sqrt(((homeostatic_dm_targets - homeostatic_dm_targets_h) ** 2).mean(axis=0))

# Get the top 10 metabolites with highest RMS
top_10_metabolites = rms_differences.nlargest(10)

print("Top 10 metabolites by RMS difference:")
print(top_10_metabolites)

# If you want just the metabolite names as a list
top_10_names = top_10_metabolites.index.tolist()
print("\nMetabolite names:")
print(top_10_names)

Top 10 metabolites by RMS difference:
WATER[c]              0.309309
CPD-12261[p]          0.019240
PROTON[c]             0.002315
PPI[c]                0.001682
LEU[c]                0.001004
L-ALPHA-ALANINE[c]    0.000964
GLY[c]                0.000764
GLT[c]                0.000759
VAL[c]                0.000758
AMP[c]                0.000721
dtype: float64

Metabolite names:
['WATER[c]', 'CPD-12261[p]', 'PROTON[c]', 'PPI[c]', 'LEU[c]', 'L-ALPHA-ALANINE[c]', 'GLY[c]', 'GLT[c]', 'VAL[c]', 'AMP[c]']


In [28]:
import plotly.graph_objects as go

# Get the top 1 metabolite
top_1_metabolite = rms_differences.nlargest(1).index[0]

# Extract the data for this metabolite from both dataframes
metabolite_data_h = homeostatic_dm_targets_h[top_1_metabolite]
metabolite_data = homeostatic_dm_targets[top_1_metabolite]

# Create the plot
fig = go.Figure()

# Add trace for simulation 1 (homeostatic_metabolite_counts_h)
fig.add_trace(go.Scatter(
    x=list(range(len(metabolite_data_h))),
    y=metabolite_data_h,
    mode='lines',
    name='Simulation Homeostatic Only',
    line=dict(width=2)  # Pastel red/pink
))

# Add trace for simulation 2 (homeostatic_metabolite_counts)
fig.add_trace(go.Scatter(
    x=list(range(len(metabolite_data))),
    y=metabolite_data,
    mode='lines',
    name='Simulation Homeostatic + Kinetics 315',
    line=dict(width=2)  # Pastel blue
))

# Update layout
fig.update_layout(
    title=f'Target dmdt of {top_1_metabolite} Across Timesteps',
    xaxis_title='Timestep',
    yaxis_title='Concentration',
    colorway=px.colors.qualitative.Pastel,
    template='plotly_white',
    hovermode='x unified',
    font=dict(size=12),
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)

fig.show()
# fig.write_image(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_kinetics/'
#                 f'dmdt_target_analysis/Water_dmdt_sim_comparison.png', width=1200, height=600, scale=3)

In [146]:
# Calculate RMS difference for each metabolite (column)
rms_differences = np.sqrt(((homeostatic_metabolite_counts - homeostatic_metabolite_counts_h) ** 2).mean(axis=0))

# Get the top 10 metabolites with highest RMS
top_10_metabolites = rms_differences.nlargest(10)

print("Top 10 metabolites by RMS difference:")
print(top_10_metabolites)

# If you want just the metabolite names as a list
top_10_names = top_10_metabolites.index.tolist()
print("\nMetabolite names:")
print(top_10_names)

Top 10 metabolites by RMS difference:
WATER[c]              0.309309
CPD-12261[p]          0.019240
PROTON[c]             0.002315
PPI[c]                0.001682
LEU[c]                0.001004
L-ALPHA-ALANINE[c]    0.000964
GLY[c]                0.000764
GLT[c]                0.000759
VAL[c]                0.000758
AMP[c]                0.000721
dtype: float64

Metabolite names:
['WATER[c]', 'CPD-12261[p]', 'PROTON[c]', 'PPI[c]', 'LEU[c]', 'L-ALPHA-ALANINE[c]', 'GLY[c]', 'GLT[c]', 'VAL[c]', 'AMP[c]']


In [48]:
import plotly.graph_objects as go
import plotly.express.colors as pc

# Get the top 1 metabolite
top_1_metabolite = rms_differences.nlargest(1).index[0]

# Get target homeostatic metabolite concentration, which is the same across the two sims
homeostatic_concs_target_combo = pd.Series(metabolism.homeostatic_concs,
                                              index=metabolism.homeostatic_metabolites)

Water_conc_target = homeostatic_concs_target_combo.loc['WATER[c]']

# Extract the data for this metabolite from both dataframes
metabolite_data_h = homeostatic_metabolite_counts_h[top_1_metabolite]
metabolite_data = homeostatic_metabolite_counts[top_1_metabolite]

# Create the plot
fig = go.Figure()

# Add trace for simulation 1 (homeostatic_metabolite_counts_h)
fig.add_trace(go.Scatter(
    x=list(range(len(metabolite_data_h))),
    y=metabolite_data_h,
    mode='lines',
    name='Simulation Homeostatic Only',
    line=dict(width=2)  # Pastel red/pink
))

# Add trace for simulation 2 (homeostatic_metabolite_counts)
fig.add_trace(go.Scatter(
    x=list(range(len(metabolite_data))),
    y=metabolite_data,
    mode='lines',
    name='Simulation Homeostatic + Kinetics 315',
    line=dict(width=2)  # Pastel blue
))

fig.add_hline(y=Water_conc_target,
              line_color=pc.qualitative.Pastel[2],
              line_dash='dash',
              annotation_text='Target Homeostatic WATER[c] Concentration',
              annotation_position="bottom right")

# Update layout
fig.update_layout(
    title=f'Concentration of {top_1_metabolite} Across Timesteps',
    xaxis_title='Timestep',
    yaxis_title='Concentration',
    colorway=px.colors.qualitative.Pastel,
    template='plotly_white',
    hovermode='x unified',
    font=dict(size=12),
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)

fig.show()
fig.write_image(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_kinetics/'
                f'dmdt_target_analysis/Water_concentration_sim_comparison.png', width=1200, height=600, scale=3)



# I want to see if target homeostatic concentration is very different between the two sims. This param is stored in the class instead of listener

In [32]:
homeostatic_concs_target_combo = pd.Series(metabolism.homeostatic_concs,
                                              index=metabolism.homeostatic_metabolites)
homeostatic_concs_target_h_only = pd.Series(metabolism_h_only.homeostatic_concs,
                                              index=metabolism.homeostatic_metabolites)

In [33]:
target_homeostatic_dmdt_combo = -(homeostatic_metabolite_counts.loc[1:, "WATER[c]"]  -
                                  homeostatic_concs_target_combo.loc["WATER[c]"])
target_homeostatic_dmdt_h_only = -(homeostatic_metabolite_counts_h.loc[1:, "WATER[c]"]  -
                                  homeostatic_concs_target_h_only.loc["WATER[c]"])
#
# Water_target_combo = target_homeostatic_dmdt_combo.loc['WATER[c]']

In [42]:
homeostatic_concs_target_h_only.loc['WATER[c]']

np.float64(42742.15931168471)

In [36]:
homeostatic_concs_target_combo[1:] == homeostatic_concs_target_h_only[1:]

2-KETOGLUTARATE[c]       True
2-PG[c]                  True
2K-4CH3-PENTANOATE[c]    True
4-AMINO-BUTYRATE[c]      True
4-hydroxybenzoate[c]     True
                         ... 
NA+[p]                   True
OXYGEN-MOLECULE[p]       True
FE+3[p]                  True
CA+2[p]                  True
Pi[p]                    True
Length: 171, dtype: bool

In [151]:
# Create the plot
fig = go.Figure()

# Add trace for simulation 1 (homeostatic_metabolite_counts_h)
fig.add_trace(go.Scatter(
    x=list(range(len(target_homeostatic_dmdt_h_only))),
    y=target_homeostatic_dmdt_h_only,
    mode='lines',
    name='Simulation H ONLY',
    line=dict(color='#FFB3BA', width=2)  # Pastel red/pink
))

# Add trace for simulation 2 (homeostatic_metabolite_counts)
fig.add_trace(go.Scatter(
    x=list(range(len(target_homeostatic_dmdt_combo))),
    y=target_homeostatic_dmdt_combo,
    mode='lines',
    name='Simulation Combo',
    line=dict(color='#BAE1FF', width=2)  # Pastel blue
))

# Update layout
fig.update_layout(
    title=f'target dmdt of {top_1_metabolite} Across Timesteps',
    xaxis_title='Timestep',
    yaxis_title='Concentration',
    template='plotly_white',
    hovermode='x unified',
    font=dict(size=12)
)

fig.show()

In [121]:
homeostatic_dm_targets_h.loc[1:, "WATER[c]"]

1      -0.306309
2       2.150827
3       3.977974
4       5.305251
5       6.612515
         ...    
596    11.338807
597    11.428087
598    11.397153
599    11.344427
600    11.416593
Name: WATER[c], Length: 600, dtype: float64

In [124]:
np.isclose(homeostatic_dm_targets_h.loc[1:, "WATER[c]"], target_homeostatic_dmdt_h_only)

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [ ]:
t.is_dcp()

In [ ]:
homeostatic_term

In [ ]:
cp.linearize(homeostatic_term) >= 0.9

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
plt.plot(counts_to_molar[1:])
plt.show()